# Setting

## Library

In [1]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Matplotlib global settings
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 20
plt.rcParams['savefig.dpi'] = 500
plt.rc('font', family='serif')

In [3]:
# ML libraries
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

In [4]:
# Helper functions & model import
sys.path.append(os.path.join('..', 'src'))
from helper import makeSpecColors
from paths import *
from var import *
from sdtpy import *
from model import *

## Function

In [5]:
def select_uids_by_class(df, sample_number, class_col='Class', uid_col='uid', random_state=42):
    np.random.seed(random_state)
    uids_by_class = {}
    for c in df[class_col].unique():
        uids = df[df[class_col]==c][uid_col].unique()
        n_select = min(sample_number, len(uids))
        selected_uids = np.random.choice(uids, n_select, replace=False)
        uids_by_class[c] = set(selected_uids)
    return uids_by_class

In [6]:
def filter_by_selected_uids(df, uids_by_class, class_col='Class', uid_col='uid'):
    mask = np.zeros(len(df), dtype=bool)
    for c, uids in uids_by_class.items():
        mask |= ((df[class_col]==c) & (df[uid_col].isin(uids)))
    return df[mask]

In [7]:
from sklearn.model_selection import GroupShuffleSplit

def train_test_split_by_uid(df, test_size=0.2, class_col='Class', uid_col='uid', random_state=42):
    groups = df[uid_col]
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(gss.split(df, df[class_col], groups))
    return df.iloc[train_idx], df.iloc[test_idx]

## Initial Setup

In [8]:
logtxt = ""

In [9]:
path_catboost = os.path.join(MODEL, "CatBoost_Fine_Tune_40")
path_xgboost = os.path.join(MODEL, "XGBoost_Fine_Tune_40")
path_lightgbm = os.path.join(MODEL, "LightGBM_Fine_Tune_40")

model_params_dict = {
    "Catboost": os.path.join(path_catboost, "best_params_n100.yaml"),
    "XGBoost": os.path.join(path_xgboost, "best_params_n100.yaml"),
    "LightGBM": os.path.join(path_lightgbm, "best_params_n100.yaml"),
}

model_params_dict

{'Catboost': '/home/gpaek/SED-Classifier/notebook/../model/CatBoost_Fine_Tune_40/best_params_n100.yaml',
 'XGBoost': '/home/gpaek/SED-Classifier/notebook/../model/XGBoost_Fine_Tune_40/best_params_n100.yaml',
 'LightGBM': '/home/gpaek/SED-Classifier/notebook/../model/LightGBM_Fine_Tune_40/best_params_n100.yaml'}

In [10]:
# Set experiment configs
test_name = "Compare_Top3"
random_state = 42
test_size = 0.2
device_type = "cpu" # or gpu
n_jobs = 10
path_save = os.path.join(MODEL, test_name)
os.makedirs(path_save, exist_ok=True)

logtxt += "\nSet experiment configs\n"
logtxt += f"test_name: {test_name}\n"
logtxt += f"random_state: {random_state}\n"
logtxt += f"test_size: {test_size}\n"
logtxt += f"device_type: {device_type}\n"
logtxt += f"n_jobs: {n_jobs}\n"
logtxt += f"path_save: {path_save}\n"
logtxt += "\n"

- Source to Consider

In [11]:
sources_to_consider = [
	"AGN", 
	"Ia", 
	"II", 
	"Ibc", 
	"LBV", 
	"TDE", 
	"Nova", 
	"M dwarf", 
	"CV",
	"SLSN",
]
logtxt += f"\nSources to consider: {sources_to_consider}\n"

In [12]:
# path_data = os.path.join(FEATURE_BALANCED_DATA, 'features_40.csv')
path_data = os.path.join(FEATURE_NEW_DATA, 'features_40_color_only.csv')

logtxt += f"\nBalanced Data Set\n"

In [13]:
path_save = os.path.join(MODEL, "three_top_models")
os.makedirs(path_save, exist_ok=True)

# Data

In [14]:
columns_to_use = list(data_dtype_dict.keys())

In [15]:
data = pd.read_csv(
    path_data,
    engine='c', 
    usecols=columns_to_use,
    dtype=data_dtype_dict,
)

data['uid'] = data['uid'].astype(str)
data['Class'] = data['Class'].astype(str)
print(f"Balanced Data: {len(data)}")

logtxt += f"Balanced Data: {len(data)}\n"

indx_type_to_consider = np.where(
	np.array([(data['Class'] == source) for source in sources_to_consider]).any(axis=0)
)

print(f"{len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}")
data = data.iloc[indx_type_to_consider[0]]

logtxt += f"Balanced: {len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}\n"
logtxt += "\n"


Balanced Data: 80605
10 sources to consider: 61233


- Training and Test Data

In [16]:
# - Split features/target
X = data.drop(columns=['Sample_ID', 'Class', 'uid'])
X.fillna(-99, inplace=True)
y = data['Class']
uids = data['uid']


# - Split into train/test using GroupShuffleSplit by uid
gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
train_idx, test_idx = next(gss.split(X, y, groups=data['uid']))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

from catboost import CatBoostClassifier, Pool

# - Label encode class for ML
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)
y_encoded = label_encoder.transform(y)
class_names = np.array([str(c) for c in label_encoder.inverse_transform(np.arange(len(label_encoder.classes_)))])
print("Balanced: Class mapping:", class_names)


Balanced: Class mapping: ['AGN' 'II' 'Ia' 'Ibc' 'SLSN' 'TDE']


In [17]:
from catboost import CatBoostClassifier, Pool
# CatBoost
# cat_train_data = Pool(data=X_train, label=y_train_encoded)
# cat_test_data = Pool(data=X_test, label=y_test_encoded)

import lightgbm as lgb
# lgb_train_data = lgb.Dataset(X_train, label=y_train_encoded)
# lgb_test_data = lgb.Dataset(X_test, label=y_test_encoded, reference=train_data)

import xgboost as xgb
# xgb_train_data = xgb.DMatrix(X_train, label=y_train_encoded)
# xgb_test_data = xgb.DMatrix(X_test, label=y_test_encoded)

In [18]:
del data

# Model

In [19]:
import time
from sklearn.metrics import mean_squared_error
from sklearn.metrics import f1_score
from xgboost.callback import EarlyStopping


from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [20]:
import numpy as np
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib

class EnsembleVotingClassifier:
    def __init__(self, 
                 catboost_model_path,
                 xgboost_model_path,
                 lightgbm_model_path,
                 weights=None):
        """
        Args:
            catboost_model_path: path to CatBoost model (.cbm)
            xgboost_model_path: path to XGBoost model (.json or .bst)
            lightgbm_model_path: path to LightGBM model (.pkl)
            weights: list or tuple of weights [cat, xgb, lgb]; if None → soft voting
        """
        self.cat = CatBoostClassifier()
        self.cat.load_model(catboost_model_path)
        
        self.xgb = XGBClassifier()
        self.xgb.load_model(xgboost_model_path)
        
        self.lgb = joblib.load(lightgbm_model_path)

        if weights is not None:
            assert len(weights) == 3, "weights must have 3 elements"
            self.weights = np.array(weights) / np.sum(weights)
        else:
            self.weights = None

    def predict_proba(self, X):
        p_cat = self.cat.predict_proba(X)
        p_xgb = self.xgb.predict_proba(X)
        p_lgb = self.lgb.predict_proba(X)

        if self.weights is None:
            # Soft voting
            probs = (p_cat + p_xgb + p_lgb) / 3
        else:
            probs = (
                p_cat * self.weights[0] +
                p_xgb * self.weights[1] +
                p_lgb * self.weights[2]
            )
        return probs

    def predict(self, X):
        probs = self.predict_proba(X)
        return np.argmax(probs, axis=1)

In [21]:
path_final_models = os.path.join(MODEL, "final_normal_class_model")

In [22]:
import json

path_weight = os.path.join(path_final_models, "ensemble_weights.json")
with open(path_weight, "r") as f:
    weight_data = json.load(f)
weights = weight_data["weights"]
print("Loaded weights:", weights)

Loaded weights: [0.12, 0.42, 0.46]


In [23]:
# 모델 로드 및 앙상블 분류기 생성
clf = EnsembleVotingClassifier(
    os.path.join(path_final_models, "catboost_model.cbm"),
    os.path.join(path_final_models, "xgboost_model.json"),
    os.path.join(path_final_models, "lightgbm_model.pkl"),
    weights=weights # None → soft voting
)

# 예측
y_pred = clf.predict(X_test)
y_probs = clf.predict_proba(X_test)

TBB Warning: The number of workers is currently limited to 19. The request for 39 workers is ignored. Further requests for more workers will be silently ignored until the limit changes.



## Single Models

In [24]:
path_cat_model = os.path.join(path_final_models, "catboost_model.cbm")
path_xgb_model = os.path.join(path_final_models, "xgboost_model.json")
path_lgb_model = os.path.join(path_final_models, "lightgbm_model.pkl")

cat = CatBoostClassifier()
cat.load_model(path_cat_model)

xgb = XGBClassifier()
xgb.load_model(path_xgb_model)

lgb = joblib.load(path_lgb_model)

In [25]:
cat_pred = cat.predict(X_test)
cat_prob = cat.predict_proba(X_test)

xgb_pred = xgb.predict(X_test)
xgb_prob = xgb.predict_proba(X_test)

lgb_pred = lgb.predict(X_test)
lgb_prob = lgb.predict_proba(X_test)

In [26]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

In [27]:
def get_report_table(model_pred):
    from sklearn.metrics import classification_report
    import pandas as pd

    # 1. report 딕셔너리로 받기
    report_dict = classification_report(y_test_encoded, model_pred, target_names=class_names, output_dict=True)

    # 2. accuracy는 scalar → 딕셔너리로 따로 저장
    acc_value = report_dict.pop("accuracy")

    # 3. DataFrame으로 변환
    report_df = pd.DataFrame(report_dict).T  # class별 성능 포함

    # 4. accuracy row 추가 (f1-score 칸만 채움)
    report_df.loc["accuracy"] = [None, None, acc_value, report_df["support"].sum()]

    return report_df

In [28]:
cat_report_df = get_report_table(cat_pred)
xgb_report_df = get_report_table(xgb_pred)
lgb_report_df = get_report_table(lgb_pred)

ValueError: Number of classes, 10, does not match size of target_names, 6. Try specifying the labels parameter

# Ensemble

## Soft Vote

In [ ]:
soft_ensemble_probs = (cat_prob + xgb_prob + lgb_prob) / 3
soft_ensemble_pred = np.argmax(soft_ensemble_probs, axis=1)

soft_ensemble_acc = accuracy_score(y_test_encoded, soft_ensemble_pred)
soft_ensemble_f1 = f1_score(y_test_encoded, soft_ensemble_pred, average='macro')

print("soft_ensemble Accuracy:", soft_ensemble_acc)
print("soft_ensemble Macro F1:", soft_ensemble_f1)
print(classification_report(y_test_encoded, soft_ensemble_pred, target_names=class_names))

In [ ]:
soft_report_df = get_report_table(soft_ensemble_pred)
soft_report_df

## Weight

- Grid Search

In [ ]:
# from sklearn.metrics import f1_score, accuracy_score

# best_f1 = 0
# best_weights = None
# best_pred = None

# weight_steps = np.linspace(0, 1, 101)  # 0.0 ~ 1.0, 0.1 간격

# for w1 in weight_steps:
#     for w2 in weight_steps:
#         w3 = 1.0 - w1 - w2
#         if w3 < 0 or w3 > 1:
#             continue  # 가중치의 합이 1이 넘어가거나 음수면 패스

#         weights = np.array([w1, w2, w3])
#         # 앙상블 확률 계산
#         ensemble_probs = (cat_prob * w1) + (xgb_prob * w2) + (lgb_prob * w3)
#         ensemble_pred = np.argmax(ensemble_probs, axis=1)
#         macro_f1 = f1_score(y_test_encoded, ensemble_pred, average='macro')

#         if macro_f1 > best_f1:
#             best_f1 = macro_f1
#             best_weights = weights
#             best_pred = ensemble_pred

# print(f"Best macro_f1: {best_f1:.4f}  |  Best weights (CatBoost, XGBoost, LightGBM): {best_weights}")

best_weights = [0.11, 0.42, 0.47]

In [ ]:
# 예: XGBoost가 가장 macro_f1이 높으니 더 높은 가중치 부여
weights = best_weights  # CatBoost, XGBoost, LightGBM 순서

weight_ensemble_probs = (
    cat_prob * weights[0] +
    xgb_prob * weights[1] +
    lgb_prob * weights[2]
)
weight_ensemble_pred = np.argmax(weight_ensemble_probs, axis=1)

weight_ensemble_acc = accuracy_score(y_test_encoded, weight_ensemble_pred)
weight_ensemble_f1 = f1_score(y_test_encoded, weight_ensemble_pred, average='macro')

# 성능 평가
from sklearn.metrics import classification_report, accuracy_score, f1_score
print("Weight Ensemble Accuracy:", accuracy_score(y_test_encoded, weight_ensemble_pred))
print("Weight Ensemble Macro F1:", f1_score(y_test_encoded, weight_ensemble_pred, average='macro'))
print(classification_report(y_test_encoded, weight_ensemble_pred, target_names=class_names))

weight_report_df = get_report_table(weight_ensemble_pred)


## Weight (class-wise)

In [ ]:
# 세 모델의 예측 확률
# # Retrieve prediction probabilities for each model
# # cat_prob = results_dict['CatBoost']['proba']
# # xgb_prob = results_dict['XGBoost']['proba']
# # lgb_prob = results_dict['LightGBM']['proba']
# # num_classes = cat_prob.shape[1] # Number of classes
# num_classes = len(class_names)

# # Dictionary to store optimal weights for each class
# # Format: {class_idx: [w_cat, w_xgb, w_lgbm]}
# best_class_weights = {}

# # Define the step size for weight grid search
# # Using 11 steps (0.0 to 1.0 with 0.1 intervals) for demonstration
# # For finer tuning, consider `np.linspace(0, 1, 51)` as in the original code
# weight_steps = np.linspace(0, 1, 101)

# print(f"\n--- Starting Class-wise Weighted Ensemble Optimization ({num_classes} classes) ---")

# # Iterate through each class to find its optimal weights
# for class_idx in range(num_classes):
#     print(f"\nOptimizing weights for Class {class_idx} ({class_names[class_idx]})")
#     best_f1_for_class = -1
#     current_class_best_weights = None

#     # Create a binary true label array for the current class
#     # 1 if the sample belongs to this class, 0 otherwise
#     y_true_binary = (y_test_encoded == class_idx).astype(int)

#     # Grid search for weights for the current class
#     for w1 in weight_steps: # Weight for CatBoost
#         for w2 in weight_steps: # Weight for XGBoost
#             w3 = 1.0 - w1 - w2 # Weight for LightGBM

#             # Ensure weights sum to 1 and are within [0, 1]
#             if w3 < 0 - 1e-9 or w3 > 1 + 1e-9: # Add tolerance for floating point comparisons
#                 continue

#             weights = np.array([w1, w2, w3])

#             # Calculate the ensemble probability for the current class
#             # This uses only the probability column corresponding to the current class
#             ensemble_prob_for_class = (
#                 cat_prob[:, class_idx] * w1 +
#                 xgb_prob[:, class_idx] * w2 +
#                 lgb_prob[:, class_idx] * w3
#             )

#             # Convert probabilities to binary predictions for F1-score calculation
#             # A threshold of 0.5 is used here; advanced tuning might involve optimizing this threshold
#             ensemble_pred_binary = (ensemble_prob_for_class >= 0.5).astype(int)

#             # Calculate F1-score for the current class (binary classification perspective)
#             f1_for_class = f1_score(y_true_binary, ensemble_pred_binary, average='binary', pos_label=1)

#             # Update best weights if current combination yields a higher F1-score
#             if f1_for_class > best_f1_for_class:
#                 best_f1_for_class = f1_for_class
#                 current_class_best_weights = weights

#     # Store the best weights found for the current class
#     best_class_weights[class_idx] = current_class_best_weights
#     print(f"  Best F1 for Class {class_idx} ({class_names[class_idx]}): {best_f1_for_class:.4f}")
#     print(f"  Best weights (CatBoost, XGBoost, LightGBM): {current_class_best_weights}")



In [ ]:
# # 세 모델의 예측 확률
# # Retrieve prediction probabilities for each model
# # cat_prob = results_dict['CatBoost']['proba']
# # xgb_prob = results_dict['XGBoost']['proba']
# # lgb_prob = results_dict['LightGBM']['proba']
# # num_classes = cat_prob.shape[1] # Number of classes
# num_classes = len(class_names)

# # Dictionary to store optimal weights for each class
# # Format: {class_idx: [w_cat, w_xgb, w_lgbm]}
# best_class_weights = {}

# # Define the step size for weight grid search
# # Using 11 steps (0.0 to 1.0 with 0.1 intervals) for demonstration
# # For finer tuning, consider `np.linspace(0, 1, 51)` as in the original code
# weight_steps = np.linspace(0, 1, 101)

# print(f"\n--- Starting Class-wise Weighted Ensemble Optimization ({num_classes} classes) ---")

# # Iterate through each class to find its optimal weights
# for class_idx in range(num_classes):
#     print(f"\nOptimizing weights for Class {class_idx} ({class_names[class_idx]})")
#     best_f1_for_class = -1
#     current_class_best_weights = None

#     # Create a binary true label array for the current class
#     # 1 if the sample belongs to this class, 0 otherwise
#     y_true_binary = (y_test_encoded == class_idx).astype(int)

#     # Grid search for weights for the current class
#     for w1 in weight_steps: # Weight for CatBoost
#         for w2 in weight_steps: # Weight for XGBoost
#             w3 = 1.0 - w1 - w2 # Weight for LightGBM

#             # Ensure weights sum to 1 and are within [0, 1]
#             if w3 < 0 - 1e-9 or w3 > 1 + 1e-9: # Add tolerance for floating point comparisons
#                 continue

#             weights = np.array([w1, w2, w3])

#             # Calculate the ensemble probability for the current class
#             # This uses only the probability column corresponding to the current class
#             ensemble_prob_for_class = (
#                 cat_prob[:, class_idx] * w1 +
#                 xgb_prob[:, class_idx] * w2 +
#                 lgb_prob[:, class_idx] * w3
#             )

#             # Convert probabilities to binary predictions for F1-score calculation
#             # A threshold of 0.5 is used here; advanced tuning might involve optimizing this threshold
#             ensemble_pred_binary = (ensemble_prob_for_class >= 0.5).astype(int)

#             # Calculate F1-score for the current class (binary classification perspective)
#             f1_for_class = f1_score(y_true_binary, ensemble_pred_binary, average='binary', pos_label=1)

#             # Update best weights if current combination yields a higher F1-score
#             if f1_for_class > best_f1_for_class:
#                 best_f1_for_class = f1_for_class
#                 current_class_best_weights = weights

#     # Store the best weights found for the current class
#     best_class_weights[class_idx] = current_class_best_weights
#     print(f"  Best F1 for Class {class_idx} ({class_names[class_idx]}): {best_f1_for_class:.4f}")
#     print(f"  Best weights (CatBoost, XGBoost, LightGBM): {current_class_best_weights}")



In [ ]:
# Class index to name mapping
# class_names = ['AGN', 'CV', 'II', 'Ia', 'Ibc', 'LBV', 'M dwarf', 'Nova', 'SLSN', 'TDE']

best_class_weights = {
    0: np.array([0.68, 0.28, 0.04]),  # AGN
    1: np.array([0.03, 0.66, 0.31]),  # CV
    2: np.array([0.60, 0.25, 0.15]),  # II
    3: np.array([0.34, 0.37, 0.29]),  # Ia
    4: np.array([0.58, 0.36, 0.06]),  # Ibc
    5: np.array([0.24, 0.22, 0.54]),  # LBV
    6: np.array([0.00, 0.19, 0.81]),  # M dwarf
    7: np.array([0.00, 0.00, 1.00]),  # Nova
    8: np.array([0.02, 0.94, 0.04]),  # SLSN
    9: np.array([0.54, 0.32, 0.14]),  # TDE
}


In [ ]:
num_classes = len(class_names)


# --- Final Ensemble Prediction using Class-wise Weights ---
# Initialize an array to store the final combined probabilities for each sample across all classes
final_ensemble_probs = np.zeros_like(cat_prob, dtype=float)

# For each sample, and for each class, apply the class-specific weights
for i in range(len(y_test_encoded)): # Iterate through each sample
    for class_idx in range(num_classes): # Iterate through each class
        # Get the optimal weights determined for this specific class
        weights_for_this_class = best_class_weights[class_idx]

        # Calculate the weighted ensemble probability for the current sample and class
        final_ensemble_probs[i, class_idx] = (
            cat_prob[i, class_idx] * weights_for_this_class[0] +
            xgb_prob[i, class_idx] * weights_for_this_class[1] +
            lgb_prob[i, class_idx] * weights_for_this_class[2]
        )

# Determine the final predicted class by selecting the one with the highest probability
final_ensemble_pred = np.argmax(final_ensemble_probs, axis=1)

class_ensemble_df = get_report_table(final_ensemble_pred)

In [ ]:

### Performance Evaluation of Class-wise Weighted Ensemble

from sklearn.metrics import classification_report, accuracy_score, f1_score

print("\n--- Final Class-wise Weighted Ensemble Performance ---")
final_ensemble_acc = accuracy_score(y_test_encoded, final_ensemble_pred)
final_ensemble_f1 = f1_score(y_test_encoded, final_ensemble_pred, average='macro')

print(f"Class-wise Weighted Ensemble Accuracy: {final_ensemble_acc:.4f}")
print(f"Class-wise Weighted Ensemble Macro F1: {final_ensemble_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_encoded, final_ensemble_pred, target_names=class_names))

print("\n--- Best Class-wise Weights Summary ---")
for class_idx, weights in best_class_weights.items():
    print(f"Class {class_idx} ({class_names[class_idx]}): CatBoost={weights[0]:.2f}, XGBoost={weights[1]:.2f}, LightGBM={weights[2]:.2f}")

# Update the log file with the new ensemble results
logtxt += "\n--- Class-wise Weighted Ensemble Results ---\n"
logtxt += f"Class-wise Weighted Ensemble Accuracy: {final_ensemble_acc:.4f}\n"
logtxt += f"Class-wise Weighted Ensemble Macro F1: {final_ensemble_f1:.4f}\n"
logtxt += classification_report(y_test_encoded, final_ensemble_pred, target_names=class_names)
logtxt += "\n"

## Combined Results

In [ ]:
result_dict = {
    "CatBoost": cat_report_df,
    "LightGBM": lgb_report_df,
    "XGBoost": xgb_report_df,
    "Soft": soft_report_df,
    "Weighted": weight_report_df,
    "Class-weighted": class_ensemble_df,
}

In [ ]:
cat_report_df

In [ ]:
import seaborn as sns

models = list(result_dict.keys())

# Define the categories for plotting (individual classes + macro avg)
kinds = list(class_names) + ['macro avg']
n_kinds = len(kinds)
metric_key = 'f1-score' # The metric we want to visualize
n_models = len(models)

colors = sns.color_palette('Paired', n_models)


# Set up the figure and axes for the bar graph
fig, ax = plt.subplots(figsize=(12, 6)) # Increased figure size for better readability

# Define bar width for each model within a 'kind' group
# Adjust width based on the number of models to prevent overlap
total_width = 0.8
single_bar_width = total_width / n_models
x_base = np.arange(n_kinds) # Base x-positions for each 'kind' group

for i, modelname in enumerate(models):
    offset = (i - (n_models - 1) / 2) * single_bar_width
    scores = [result_dict[modelname][metric_key][kind] for kind in kinds]

    # rects = ax.bar(x_base + offset, scores, single_bar_width, label=modelname, color=colors[i])
    # 각 모델별로 다른 색상을 할당합니다.
    ax.bar(x_base + offset, scores, single_bar_width, label=modelname, color=colors[i])

ax.set_xlabel('Class / Average Type', fontsize=14)
ax.set_ylabel('F1-score', fontsize=14)
ax.set_title('Per-Class and Macro F1-score Comparison Across Models', fontsize=16)
ax.set_xticks(x_base)
ax.set_xticklabels(kinds, rotation=45, ha='right', fontsize=12)
ax.set_ylim(0.5, 1.0)
ax.grid(axis='y', linestyle='--', alpha=0.7)

ax.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=12)

plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.show()

In [ ]:
# Macro avg F1-score만 추출
macro_f1_scores = [result_dict[model]['f1-score']['macro avg'] for model in models]

# Set up the figure and axes
fig, ax = plt.subplots(figsize=(8, 5)) # 그래프 크기 조정

# Define colors using a seaborn palette
colors = sns.color_palette('Paired', len(models)) # 모델 수에 맞는 색상 추출

# Plot the bars
bars = ax.bar(models, macro_f1_scores, color=colors)

# Add score labels on top of the bars
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 0.005, round(yval, 3), ha='center', va='bottom') # 소수점 셋째 자리까지 표시

# Set labels, title, and ticks
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Macro Avg F1-score', fontsize=12)
ax.set_title('Macro Average F1-score Comparison Across Models', fontsize=14)
ax.set_ylim(min(macro_f1_scores) * 0.95, max(macro_f1_scores) * 1.05) # Y-축 범위 자동 조정 (약간의 여백)
ax.set_ylim(0.75, 0.8) # F1-score는 0-1 사이이므로 고정할 수도 있습니다.
ax.grid(axis='y', linestyle='--', alpha=0.7) # Add horizontal grid lines

plt.tight_layout() # Adjust layout to prevent labels from overlapping
plt.show()

## Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 예시: 기존 결과 및 앙상블 결과 저장
metrics_summary = {
    'CatBoost':   {'accuracy': 0.751314, 'macro_f1': 0.752196},
    'XGBoost':    {'accuracy': 0.765804, 'macro_f1': 0.772247},
    'LightGBM':   {'accuracy': 0.760767, 'macro_f1': 0.773273},
    'Soft':   {'accuracy': ensemble_acc, 'macro_f1': ensemble_f1},  # <- 앙상블 결과 입력
    'Weighted': {'accuracy': weight_ensemble_acc, 'macro_f1': weight_ensemble_f1},
    'Class_Weighted': {'accuracy': final_ensemble_acc, 'macro_f1': final_ensemble_f1},
}

models = list(metrics_summary.keys())
accuracy = [metrics_summary[m]['accuracy'] for m in models]
macro_f1 = [metrics_summary[m]['macro_f1'] for m in models]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
rects1 = ax.bar(x - width/2, accuracy, width, label='Accuracy')
rects2 = ax.bar(x + width/2, macro_f1, width, label='Macro F1')

ax.set_ylabel('Score')
ax.set_title('Model-wise Accuracy & Macro F1-score Comparison')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend(loc='upper center', ncol=2, fontsize=14)
ax.set_ylim(0.7, 0.8)
ax.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 6))

# 기존 모델별 F1-score (예시로 per_class_df를 그대로 활용)
for model_name in ['CatBoost', 'XGBoost', 'LightGBM']:
    plt.plot(class_names, per_class_df[(model_name, 'f1-score')], marker='o', label=model_name, ls='')

# 앙상블(soft voting) F1-score 추가
from sklearn.metrics import f1_score
ensemble_f1_class = f1_score(y_test_encoded, ensemble_pred, average=None)  # array, 클래스 순서는 class_names와 같음
plt.plot(class_names, ensemble_f1_class, marker='*', label='Soft Voting', linewidth=3, ls='')

# Weight
weight_ensemble_f1_class = f1_score(y_test_encoded, weight_ensemble_pred, average=None)  # array, 클래스 순서는 class_names와 같음
plt.plot(class_names, weight_ensemble_f1_class, marker='*', label='Weighted Voting', linewidth=3, ls='')

#
class_weight_ensemble_f1_class = f1_score(y_test_encoded, final_ensemble_pred, average=None)  # array, 클래스 순서는 class_names와 같음
plt.plot(class_names, class_weight_ensemble_f1_class, marker='*', label='Class-Weighted Voting', linewidth=3, ls='')

plt.title('Per-class F1-score Comparison (Single Models vs. Ensemble)')
plt.xlabel('Class')
plt.ylabel('F1-score')
plt.legend()
plt.xticks(rotation=45)
plt.grid()
plt.tight_layout()
plt.show()


In [ ]:
improvement = np.array(ensemble_f1_class) - np.array([per_class_df[(model_name, 'f1-score')] for model_name in ['XGBoost']])[0]
plt.figure(figsize=(10, 5))
plt.bar(class_names, improvement, color='green')
plt.axhline(0, color='gray', linestyle='--')
plt.ylabel('F1-score Gain')
plt.title('Soft Voting')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
weight_improvement = np.array(weight_ensemble_f1_class) - np.array([per_class_df[(model_name, 'f1-score')] for model_name in ['XGBoost']])[0]
plt.figure(figsize=(10, 5))
plt.bar(class_names, weight_improvement, color='dodgerblue')
plt.axhline(0, color='gray', linestyle='--')
plt.ylabel('F1-score Gain')
plt.title('Weighted Voting')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# CV

In [ ]:
import yaml

# ---------------------------------------------------------
# 1. Load Best Hyperparameters from YAML
# ---------------------------------------------------------
def load_yaml_params(path_yaml):
    with open(path_yaml, "r") as f:
        params = yaml.safe_load(f)
    return params

cat_params = load_yaml_params(model_params_dict["Catboost"])
cat_params['thread_count'] = n_jobs
xgb_params = load_yaml_params(model_params_dict["XGBoost"])
xgb_params['nthread'] = n_jobs
lgbm_params = load_yaml_params(model_params_dict["LightGBM"])
lgbm_params['num_threads'] = n_jobs

logtxt += f"Loaded CatBoost params: {cat_params}\n"
logtxt += f"Loaded XGBoost params: {xgb_params}\n"
logtxt += f"Loaded LightGBM params: {lgbm_params}\n"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns # Optional, but good for aesthetics
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, Pool
from xgboost import XGBClassifier
import lightgbm as lgb # For lgb.early_stopping
import os # For saving results
groups = uids.values
gkf = GroupKFold(n_splits=5)

lgbm_macro_f1s, catb_macro_f1s, xgb_macro_f1s = [], [], []
soft_macro_f1s, weight_macro_f1s, class_weight_macro_f1s = [], [], []

# Define the step size for class-wise weight grid search
weight_steps = np.linspace(0, 1, 11) # 0.0 to 1.0 with 0.1 intervals

# Perform Grouped K-Fold Cross-Validation
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y_encoded, groups=groups)):
    print(f"\n=== Fold {fold+1} ===")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]

    # --- Individual Model Training and Prediction ---
    # LightGBM
    lgbm_model = LGBMClassifier(**lgbm_params, n_jobs=n_jobs)
    lgbm_model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                    eval_metric='multi_logloss',
                    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)])
    lgbm_pred = lgbm_model.predict(X_val)
    lgbm_prob = lgbm_model.predict_proba(X_val)
    lgbm_macro_f1s.append(f1_score(y_val, lgbm_pred, average='macro'))
    print(f"LGBM Macro F1: {lgbm_macro_f1s[-1]:.4f}")

    # CatBoost
    cat_model = CatBoostClassifier(**cat_params) # verbose=0 and early_stopping_rounds are in cat_params
    train_pool = Pool(X_train, label=y_train)
    eval_pool = Pool(X_val, label=y_val)
    cat_model.fit(train_pool, eval_set=eval_pool, early_stopping_rounds=30) # Pass early_stopping_rounds directly to fit
    cat_pred = cat_model.predict(X_val)
    cat_prob = cat_model.predict_proba(X_val)
    catb_macro_f1s.append(f1_score(y_val, cat_pred, average='macro'))
    print(f"CatBoost Macro F1: {catb_macro_f1s[-1]:.4f}")

    # XGBoost
    xgb_model = XGBClassifier(**xgb_params) # n_jobs, use_label_encoder, eval_metric, verbosity are in xgb_params
    # early_stopping_rounds와 eval_set을 fit 메서드에 직접 전달하여 오류 해결
    xgb_model.fit(X_train, y_train,
                #   early_stopping_rounds=30,
                  eval_set=[(X_val, y_val)],
                  verbose=False) # verbose=False로 설정하여 학습 과정 출력 억제
    xgb_pred = xgb_model.predict(X_val)
    xgb_prob = xgb_model.predict_proba(X_val)
    xgb_macro_f1s.append(f1_score(y_val, xgb_pred, average='macro'))
    print(f"XGBoost Macro F1: {xgb_macro_f1s[-1]:.4f}")

    # --- Ensemble Methods ---
    # Soft Voting
    soft_prob = (lgbm_prob + cat_prob + xgb_prob) / 3
    soft_pred = np.argmax(soft_prob, axis=1)
    soft_macro_f1s.append(f1_score(y_val, soft_pred, average='macro'))
    print(f"Soft Voting Macro F1: {soft_macro_f1s[-1]:.4f}")

    # Weighted Voting (using global best_weights)
    weight_prob = (lgbm_prob * best_weights[0]) + (cat_prob * best_weights[1]) + (xgb_prob * best_weights[2])
    weight_pred = np.argmax(weight_prob, axis=1)
    weight_macro_f1s.append(f1_score(y_val, weight_pred, average='macro'))
    print(f"Weighted Voting Macro F1: {weight_macro_f1s[-1]:.4f}")

    # --- Class-wise Weighted Voting ---
    num_classes = len(class_names)
    best_class_weights = {} # Store optimal weights for each class in this fold

    print(f"--- Optimizing Class-wise Weights for Fold {fold+1} ---")
    for class_idx in range(num_classes):
        y_true_binary = (y_val == class_idx).astype(int)
        best_f1_for_class = -1
        current_class_best_weights = None

        for w1 in weight_steps:
            for w2 in weight_steps:
                w3 = 1.0 - w1 - w2

                if w3 < -1e-9 or w3 > 1 + 1e-9:
                    continue

                weights = np.array([w1, w2, w3])

                ensemble_prob_for_class = (
                    cat_prob[:, class_idx] * weights[0] +
                    xgb_prob[:, class_idx] * weights[1] +
                    lgbm_prob[:, class_idx] * weights[2]
                )

                ensemble_pred_binary = (ensemble_prob_for_class >= 0.5).astype(int)

                if np.sum(y_true_binary) == 0 and np.sum(ensemble_pred_binary) == 0:
                    f1_for_class = 1.0
                elif np.sum(y_true_binary) == 0 and np.sum(ensemble_pred_binary) > 0:
                    f1_for_class = 0.0
                else:
                    f1_for_class = f1_score(y_true_binary, ensemble_pred_binary, average='binary', pos_label=1, zero_division=0)

                if f1_for_class > best_f1_for_class:
                    best_f1_for_class = f1_for_class
                    current_class_best_weights = weights

        best_class_weights[class_idx] = current_class_best_weights
    
    # Apply class-wise weights to get final ensemble predictions for this fold's validation set
    final_ensemble_probs_fold = np.zeros_like(cat_prob, dtype=float)
    for i_sample in range(len(y_val)):
        for class_idx in range(num_classes):
            weights_for_this_class = best_class_weights[class_idx]
            final_ensemble_probs_fold[i_sample, class_idx] = (
                cat_prob[i_sample, class_idx] * weights_for_this_class[0] +
                xgb_prob[i_sample, class_idx] * weights_for_this_class[1] +
                lgbm_prob[i_sample, class_idx] * weights_for_this_class[2]
            )
    final_ensemble_pred_fold = np.argmax(final_ensemble_probs_fold, axis=1)
    class_weight_macro_f1s.append(f1_score(y_val, final_ensemble_pred_fold, average='macro'))
    print(f"Class-wise Weighted Voting Macro F1: {class_weight_macro_f1s[-1]:.4f}")
    print("-" * 30)


# Create a DataFrame from the collected Macro F1-scores for CV visualization
results_df = pd.DataFrame({
    'LightGBM': lgbm_macro_f1s,
    'CatBoost': catb_macro_f1s,
    'XGBoost': xgb_macro_f1s,
    'Soft Voting': soft_macro_f1s,
    'Weighted Voting': weight_macro_f1s,
    'Class-Weighted Voting': class_weight_macro_f1s
})

# Optional: Save the results to a CSV file
# path_save = './results'
# os.makedirs(path_save, exist_ok=True)
# cv_result_path = os.path.join(path_save, 'macro_f1_cv_results.csv')
# results_df.to_csv(cv_result_path, index=False)
# print(f"\nCV results saved to: {cv_result_path}")

# --- Visualization of CV Results ---
plt.figure(figsize=(12, 7))
sns.boxplot(data=results_df, palette='viridis')

plt.ylabel('Macro F1-score', fontsize=12)
plt.title('5-fold CV Macro F1-score Comparison: Single Models vs. Ensemble Methods', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.xticks(rotation=45, ha='right', fontsize=10)

plt.tight_layout()
plt.show()

print("\nMean and Standard Deviation of Macro F1-scores across folds:")
print(results_df.agg(['mean', 'std']).round(4))

# Feature Importance

In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# 환경변수로 core 제한 (추가 안정성 확보)
os.environ["OMP_NUM_THREADS"] = "15"
os.environ["MKL_NUM_THREADS"] = "15"
os.environ["TBB_NUM_THREADS"] = "15"

# 데이터 샘플링
X_sample = X_train.sample(n=1000, random_state=42)

# SHAP explainer 계산 (TreeExplainer는 model structure 사용)
explainer_cat = shap.TreeExplainer(cat)
shap_values_cat = explainer_cat.shap_values(X_sample)
importance_cat = np.abs(shap_values_cat).mean(axis=0)

explainer_xgb = shap.TreeExplainer(xgb)
shap_values_xgb = explainer_xgb.shap_values(X_sample)
importance_xgb = np.abs(shap_values_xgb).mean(axis=0)

explainer_lgb = shap.TreeExplainer(lgb)
shap_values_lgb = explainer_lgb.shap_values(X_sample)
importance_lgb = np.abs(shap_values_lgb).mean(axis=0)


In [ ]:
mean_importance_cat = importance_cat.sum(axis=0)
mean_importance_xgb = importance_xgb.sum(axis=0)
mean_importance_lgb = importance_lgb.sum(axis=0)

In [ ]:
# Normalize
nor_mean_importance_xgb = mean_importance_xgb / mean_importance_xgb.sum()
nor_mean_importance_cat = mean_importance_cat / mean_importance_cat.sum()
nor_mean_importance_lgb = mean_importance_lgb / mean_importance_lgb.sum()

# Voting 방식
importance_soft = (nor_mean_importance_cat + nor_mean_importance_xgb + nor_mean_importance_lgb) / 3
nor_importance_soft = importance_soft / importance_soft.sum()
w_cat, w_xgb, w_lgb = best_weights
importance_weighted = (
    nor_mean_importance_cat * w_cat +
    nor_mean_importance_xgb * w_xgb +
    nor_mean_importance_lgb * w_lgb
)

nor_importance_weighted = importance_weighted / importance_weighted.sum()

In [ ]:

# DataFrame
feature_names = X_train.columns
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'CatBoost': nor_mean_importance_cat,
    'XGBoost': nor_mean_importance_xgb,
    'LightGBM': nor_mean_importance_lgb,
    'Soft Voting': nor_importance_soft,
    'Weighted Voting': nor_importance_weighted
}).set_index('Feature')


In [ ]:
# Compute class-wise weighted importance
importance_class_weighted = np.zeros_like(nor_mean_importance_cat)
for class_idx in range(len(class_names)):
    w_cat, w_xgb, w_lgb = best_class_weights[class_idx]
    imp = (
        nor_mean_importance_cat * w_cat +
        nor_mean_importance_xgb * w_xgb +
        nor_mean_importance_lgb * w_lgb
    )
    importance_class_weighted += imp

importance_class_weighted /= len(class_names)  # 평균화

# Normalize
nor_importance_class_weighted = importance_class_weighted / importance_class_weighted.sum()

# DataFrame 구성
importance_df['Class-weighted Voting'] = importance_class_weighted


# # 개별 모델 비교
# importance_df_sorted[['CatBoost', 'XGBoost', 'LightGBM']].plot.barh(stacked=False, figsize=(10, 8))
# plt.gca().invert_yaxis()
# plt.title("Top 20 Features (SHAP Importance from Individual Models)")
# plt.xlabel("Normalized Mean(|SHAP Value|)")
# plt.tight_layout()
# plt.show()

# # Ensemble 비교 (Soft, Weighted, Class-Weighted)
# importance_df_sorted[['Soft Voting', 'Weighted Voting', 'Class-weighted Voting']].plot.barh(stacked=False, figsize=(10, 8), color=['gray', 'blue', 'green'])
# plt.gca().invert_yaxis()
# plt.title("Top 20 Features (SHAP Ensemble Importance)")
# plt.xlabel("Normalized Mean(|SHAP Value|)")
# plt.tight_layout()
# plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# 설정
methods = ['CatBoost', 'XGBoost', 'LightGBM', 'Soft Voting', 'Weighted Voting', 'Class-weighted Voting']
n_methods = len(methods)
n_cols = 3
n_rows = (n_methods + n_cols - 1) // n_cols  # 올림 나눗셈
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 7 * n_rows), squeeze=False)

# 컬러맵에서 기본 색상 추출 (tab10 등 sequential colormap 활용 가능)
colors = plt.cm.tab10(np.linspace(0, 1, n_methods))

# 각 subplot에 그래프 그리기
for i, met in enumerate(methods):
    row, col = divmod(i, n_cols)
    ax = axes[row][col]

    importance_df_sorted = importance_df.sort_values(by=met, ascending=False).head(20)

    # 막대 그리기 (테두리: 검정색, 채움색: 모델별 색상)
    ax.barh(
        importance_df_sorted.index,
        importance_df_sorted[met],
        color=colors[i],
        edgecolor='black'
    )
    ax.invert_yaxis()
    ax.set_title(f"Top 20 Features ({met})", fontsize=14)
    if i == 4:
        ax.set_xlabel("Normalized Mean(|SHAP Value|)")
    ax.tick_params(axis='both', labelsize=10)

# 빈 subplot 숨기기
for i in range(len(methods), n_rows * n_cols):
    row, col = divmod(i, n_cols)
    axes[row][col].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
feature_score_table = Table()
feature_score_table['feature'] = feature_names
for filte in MEDIUM_BANDS:
    feature_score_table[filte] = 0.

In [ ]:
default_score = 1.0

weight_vote_score_table = feature_score_table.copy()
for ff, feature in enumerate(weight_vote_score_table['feature']):

    weight = nor_importance_weighted[ff]

    filter0 = feature.split('-')[0]
    filter1 = feature.split('-')[1]

    weight_vote_score_table[filter0][ff] = default_score * weight
    weight_vote_score_table[filter1][ff] = default_score * weight

weight_vote_score_table[:3]

In [ ]:
total_filter_scores = [weight_vote_score_table[filte].sum() for filte in MEDIUM_BANDS]
plt.figure(figsize=(10, 4))
plt.bar(MEDIUM_BANDS, total_filter_scores, width=0.5,)
_ = plt.xticks(rotation=90)
plt.ylabel("Importance")
plt.tight_layout()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table

def compute_filter_scores_from_feature_importance(
    feature_names,
    importance,
    medium_band_filters,
    default_score=1.0,
    normalize=True,
    plot=True,
    title="Filter-wise Importance from Feature Scores"
):
    """
    Args:
        feature_names: list of color feature names (e.g., 'm400-m425')
        importance: 1D array or list of feature importance values (length = len(feature_names))
        medium_band_filters: list of medium-band filters to accumulate importance (e.g., ['m400', 'm425', ...])
        default_score: multiplier for each feature's normalized importance (default=1.0)
        normalize: whether to normalize input importance to sum to 1
        plot: whether to show bar chart of total filter-wise importance
        title: plot title
    Returns:
        score_table: Table of importance contribution from each feature to each filter
        total_filter_scores: 1D list of total importance per filter
    """
    if normalize:
        importance = np.array(importance)
        importance = importance / importance.sum()

    # Initialize feature score table
    score_table = Table()
    score_table['feature'] = feature_names
    for filt in medium_band_filters:
        score_table[filt] = 0.0

    # Fill values
    for ff, feat in enumerate(score_table['feature']):
        filt0, filt1 = feat.split('-')
        if filt0 in medium_band_filters:
            score_table[filt0][ff] = default_score * importance[ff]
        if filt1 in medium_band_filters:
            score_table[filt1][ff] = default_score * importance[ff]

    # Total score per filter
    total_scores = [score_table[filt].sum() for filt in medium_band_filters]

    # Plot
    if plot:
        colors = makeSpecColors(len(medium_band_filters), palette='Spectral')[::-1]
        plt.figure(figsize=(10, 4))
        plt.bar(medium_band_filters, total_scores, width=0.5, color=colors, edgecolor='black')
        plt.xticks(rotation=90)
        plt.ylabel("Importance")
        plt.title(title)
        plt.tight_layout()
        plt.show()

    return score_table, total_scores

In [ ]:
norm_importances = [nor_mean_importance_xgb, nor_mean_importance_lgb, nor_mean_importance_cat, 
                    nor_importance_soft, nor_importance_weighted, nor_importance_class_weighted]

for title, importance in zip(methods, norm_importances):

    score_table, total_scores = compute_filter_scores_from_feature_importance(
        feature_names=X_train.columns,
        importance=importance,
        medium_band_filters=MEDIUM_BANDS,
        default_score=1.0,
        normalize=True,
        plot=True,
        title=title,
    )

In [ ]:
total_scores_arr = np.array(total_scores)

sorted_filters_by_importance = np.array(MEDIUM_BANDS)[np.argsort(-1*total_scores_arr)]

sorted_filters_by_importance[:20]

## Sensitivity vs Feature Importance 

In [ ]:
# 7DT Setting
sys.path.append('..')
from simulator.helper import *
from simulator.sdtpy import *
register_custom_filters_on_speclite('../simulator')

#	Exposure Time [s]
sdt = SevenDT()
sdt.echo_optics()
filterset = sdt.generate_filterset(bandmin=BANDMIN, bandmax=BANDMAX, bandwidth=BANDWIDTH, bandstep=BANDSTEP, bandrsp=BANDRSP, lammin=LAMMIN, lammax=LAMMAX, lamres=LAMRES)
T_qe = sdt.get_CMOS_IMX455_QE()
sdt.get_optics()
s = sdt.get_sky()
sdt.smooth_sky()
totrsptbl = sdt.calculate_response()
Npix_ptsrc, Narcsec_ptsrc = sdt.get_phot_aperture(exptime=EXPTIME_SINGLE, fwhm_seeing=SEEING, optfactor=EFF_FACTOR, verbose=False)
depthtbl = sdt.get_depth_table(Nsigma=5)
sdt.get_speclite()

In [ ]:
totrsptbl[:3]

In [ ]:
rsp_arr = []

for filte in np.unique(totrsptbl['name']):
    rsp = totrsptbl['response'][totrsptbl['name'] == filte]
    max_rsp = np.max(rsp)
    # print(filtername, max_rsp)
    rsp_arr.append(max_rsp)

rsp_arr = np.array(rsp_arr)

In [ ]:
colors = makeSpecColors(len(MEDIUM_BANDS), palette='Spectral')[::-1]

plt.scatter(rsp_arr, total_scores, marker='o', c=colors, ec='k')

score_threshold = 0.06
resp_threshold = 0.5
for filte, resp_value, score in zip(MEDIUM_BANDS, rsp_arr, total_scores):
    if (score > score_threshold) & (resp_value < resp_threshold):
        plt.text(resp_value, score, s=filte)

plt.xlabel("Maximum Response [%]")
plt.ylabel("Importance")

# Result Save

In [ ]:
path_final_models = os.path.join(MODEL, "final_normal_class_model")
os.makedirs(path_final_models, exist_ok=True)

In [ ]:
# CatBoost
cat_model.save_model(os.path.join(path_final_models, "catboost_model.cbm"))

# XGBoost
xgb_model.save_model(os.path.join(path_final_models, "xgboost_model.json"))  # or .bst

# LightGBM
import joblib
joblib.dump(lgbm_model, os.path.join(path_final_models, "lightgbm_model.pkl"))

In [ ]:
import numpy as np
import joblib
import shap
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# 모델 불러오기
def load_ensemble_models(cat_path, xgb_path, lgb_path):
    cat = CatBoostClassifier()
    cat.load_model(cat_path)

    xgb = XGBClassifier()
    xgb.load_model(xgb_path)

    lgb = joblib.load(lgb_path)

    return cat, xgb, lgb

# 예측 함수 정의
def ensemble_predict(X, cat, xgb, lgb, weights=None):
    """
    Args:
        X: input features (DataFrame or ndarray)
        cat, xgb, lgb: trained model instances
        weights: None for soft voting, or list like [w_cat, w_xgb, w_lgb]
    Returns:
        pred_class: final predicted class index
        pred_probs: averaged or weighted probability
    """
    p_cat = cat.predict_proba(X)
    p_xgb = xgb.predict_proba(X)
    p_lgb = lgb.predict_proba(X)

    if weights is None:
        # soft voting
        probs = (p_cat + p_xgb + p_lgb) / 3
    else:
        probs = (
            p_cat * weights[0] +
            p_xgb * weights[1] +
            p_lgb * weights[2]
        )

    pred_class = np.argmax(probs, axis=1)
    return pred_class, probs

In [ ]:
# Load models
cat, xgb, lgb = load_ensemble_models(
    os.path.join(path_final_models, "catboost_model.cbm"),
    os.path.join(path_final_models, "xgboost_model.json"),
    os.path.join(path_final_models, "lightgbm_model.pkl"),
)

# Predict
y_pred, y_probs = ensemble_predict(X_test, cat, xgb, lgb, weights=best_weights)

In [ ]:
import json
json.dump({"weights": [val for val in best_weights]}, open(os.path.join(path_final_models, "ensemble_weights.json"), "w"))